# Native Sparse Attention (NSA) — a toy-scale build

A minimal implementation of **Native Sparse Attention**, from DeepSeek-AI,
*"Native Sparse Attention: Hardware-Aligned and Natively Trainable Sparse
Attention"* (2025) — a way to make attention cheaper by making it
*trainably sparse*, rather than dense-then-pruned.

Companion write-up: `README.md` in this folder.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 1. The idea

Regular causal attention makes every token attend to *every* earlier token —
that's what makes it expensive at long context. A lot of prior work tries to
sparsify attention *after* training (only look at a subset of tokens), but
that tends to hurt quality, because the model was never trained to work
with sparse attention patterns in the first place.

NSA's approach: build the sparsity **into the architecture from the start**,
so the model learns to use it well. Every token's attention output is a
learned combination of **three branches**, each attending differently:

| branch | what it attends to | good at |
|---|---|---|
| **compressed** | coarse summaries of *blocks* of tokens (mean-pooled) | cheap, global context |
| **selected** | full-resolution attention, but only within the *few* blocks the compressed branch found most relevant | precise long-range recall, without the full cost |
| **sliding window** | a plain local window of the most recent tokens | fine-grained local/syntactic patterns |

The three branches share the same Q/K/V projections, run independently, and
get combined with a **learned, per-token, per-head gate**.

## 2. Each branch, step by step

**Compressed branch:** split the sequence into fixed-size blocks and
mean-pool each block's keys and values into one summary vector per block.
Run ordinary attention from every query to these (much fewer) block
summaries. This is cheap because there are far fewer blocks than tokens,
and it gives every query at least a coarse view of the whole sequence.

**Selected branch:** reuse the *attention scores* the compressed branch just
computed as an importance signal — "which blocks did this query find most
relevant?" Pick the top-`n` blocks per query, then run **full-resolution**
attention against every individual token inside just those blocks. This is
where NSA gets precise, without paying for full-resolution attention against
every token in the sequence.

**Sliding window branch:** completely ordinary causal attention, just capped
to a fixed window of the most recent tokens. This branch exists mostly so
the model always has cheap, reliable access to local context, which the
other two branches aren't well-suited for on their own.

**Combining:** a small linear layer on the input predicts one gate value per
branch, per head, per token (via sigmoid); the final output is the
gate-weighted sum of the three branches' outputs.

> **Simplification used here:** the paper's actual speedup comes from a
> custom, hardware-aligned Triton kernel that exploits the sparsity pattern
> directly (skipping the un-selected blocks entirely rather than computing
> and discarding them). This notebook computes the compressed and
> local branches densely and only truly narrows compute in the *selected*
> branch (via `gather`), since that's enough to demonstrate the mechanism's
> actual sparsity pattern and causal masking correctly at toy scale — a real
> implementation would skip far more compute than this one does.

In [ ]:
class NSA(nn.Module):
    def __init__(self, d_model=64, n_heads=2, d_head=32, block_size=4, n_select=2, window=4):
        super().__init__()
        self.h, self.dh, self.bs = n_heads, d_head, block_size
        self.nsel, self.win = n_select, window
        inner = n_heads * d_head
        self.q_proj = nn.Linear(d_model, inner, bias=False)
        self.k_proj = nn.Linear(d_model, inner, bias=False)
        self.v_proj = nn.Linear(d_model, inner, bias=False)
        self.gate_proj = nn.Linear(d_model, 3 * n_heads, bias=True)   # per-branch, per-head gate
        self.out_proj = nn.Linear(inner, d_model, bias=False)
        self.scale = d_head ** -0.5

    def forward(self, x):
        B, T, D = x.shape
        H, Dh, bs = self.h, self.dh, self.bs
        device = x.device
        q = self.q_proj(x).view(B, T, H, Dh).transpose(1, 2)
        k = self.k_proj(x).view(B, T, H, Dh).transpose(1, 2)
        v = self.v_proj(x).view(B, T, H, Dh).transpose(1, 2)
        causal = torch.triu(torch.ones(T, T, device=device, dtype=torch.bool), diagonal=1)

        # --- branch 1: compressed (block-mean-pooled K/V, dense attention over block summaries) ---
        n_blocks = (T + bs - 1) // bs
        pad = n_blocks * bs - T
        k_pad = F.pad(k, (0, 0, 0, pad))
        v_pad = F.pad(v, (0, 0, 0, pad))
        k_blk = k_pad.view(B, H, n_blocks, bs, Dh).mean(3)      # block summaries
        v_blk = v_pad.view(B, H, n_blocks, bs, Dh).mean(3)
        blk_start = torch.arange(n_blocks, device=device) * bs
        blk_future = blk_start.view(1, -1) > torch.arange(T, device=device).view(-1, 1)  # block hasn't started yet
        scores_c = torch.einsum('bhtd,bhkd->bhtk', q, k_blk) * self.scale
        scores_c = scores_c.masked_fill(blk_future.view(1, 1, T, n_blocks), float('-inf'))
        attn_c = scores_c.softmax(-1)
        out_c = torch.einsum('bhtk,bhkd->bhtd', attn_c, v_blk)

        # --- branch 2: selected (top-n most important blocks by branch-1 scores, full-res attention) ---
        importance = scores_c.masked_fill(torch.isinf(scores_c), -1e9)
        nsel = min(self.nsel, n_blocks)
        top_idx = importance.topk(nsel, dim=-1).indices                  # B,H,T,nsel -- which blocks per query
        k_pad_r = k_pad.view(B, H, n_blocks, bs, Dh)
        v_pad_r = v_pad.view(B, H, n_blocks, bs, Dh)
        gather_idx = top_idx.view(B, H, T, nsel, 1, 1).expand(-1, -1, -1, -1, bs, Dh)
        k_sel = k_pad_r.unsqueeze(2).expand(-1, -1, T, -1, -1, -1).gather(3, gather_idx).reshape(B, H, T, nsel * bs, Dh)
        v_sel = v_pad_r.unsqueeze(2).expand(-1, -1, T, -1, -1, -1).gather(3, gather_idx).reshape(B, H, T, nsel * bs, Dh)
        sel_pos = (top_idx.unsqueeze(-1) * bs + torch.arange(bs, device=device).view(1, 1, 1, 1, -1)).reshape(B, H, T, nsel * bs)
        q_pos = torch.arange(T, device=device).view(1, 1, T, 1)
        sel_mask = (sel_pos > q_pos) | (sel_pos >= T)
        scores_s = torch.einsum('bhtd,bhtkd->bhtk', q, k_sel) * self.scale
        scores_s = scores_s.masked_fill(sel_mask, float('-inf'))
        attn_s = scores_s.softmax(-1)
        out_s = torch.einsum('bhtk,bhtkd->bhtd', attn_s, v_sel)

        # --- branch 3: sliding window (local causal attention, last `window` tokens) ---
        scores_w = torch.einsum('bhtd,bhsd->bhts', q, k) * self.scale
        pos_diff = torch.arange(T, device=device).view(-1, 1) - torch.arange(T, device=device).view(1, -1)
        win_mask = causal | (pos_diff >= self.win)
        attn_w = scores_w.masked_fill(win_mask, float('-inf')).softmax(-1)
        out_w = torch.einsum('bhts,bhsd->bhtd', attn_w, v)

        # --- combine branches with a learned, per-token, per-head gate ---
        gates = torch.sigmoid(self.gate_proj(x)).view(B, T, H, 3).permute(0, 2, 1, 3)   # B,H,T,3
        combined = gates[..., 0:1] * out_c + gates[..., 1:2] * out_s + gates[..., 2:3] * out_w
        combined = combined.transpose(1, 2).reshape(B, T, H * Dh)
        return self.out_proj(combined)

## 3. Assembling a tiny language model

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(norm + self.eps) * self.weight

class SwiGLU(nn.Module):
    def __init__(self, d, hidden_mult=2):
        super().__init__()
        h = d * hidden_mult
        self.Wg = nn.Linear(d, h, bias=False)
        self.Wu = nn.Linear(d, h, bias=False)
        self.Wd = nn.Linear(h, d, bias=False)
    def forward(self, x):
        return self.Wd(F.silu(self.Wg(x)) * self.Wu(x))

class TinyLM(nn.Module):
    def __init__(self, vocab_size, d_model=64, n_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([NSA(d_model) for _ in range(n_layers)])
        self.mlps = nn.ModuleList([SwiGLU(d_model) for _ in range(n_layers)])
        self.norms1 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.norms2 = nn.ModuleList([RMSNorm(d_model) for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, idx):
        x = self.embed(idx)
        for blk, mlp, n1, n2 in zip(self.blocks, self.mlps, self.norms1, self.norms2):
            x = x + blk(n1(x))
            x = x + mlp(n2(x))
        return self.lm_head(self.final_norm(x))

## Proving it actually works

Everything above is only worth something if gradients actually flow correctly
through NSA once it's wired into a real model. So the rest of this
notebook:

1. wraps NSA into a tiny 2-layer causal language model,
2. builds a **tiny synthetic dataset** (a repeating `"0123456789ABCDEF"`
   string — enough to check the model can learn *any* sequential structure
   at all, no real corpus needed),
3. runs **one forward + backward pass** as a sanity check (right output
   shape, no `NaN` gradients),
4. **trains for a few hundred steps**, and
5. **generates** from the trained model — if training worked, the output
   should show visible periodicity.

This is deliberately not a "real" training run. It exists purely to catch
architecture bugs, which is the whole point of a toy-scale build.

In [ ]:
# --- synthetic dataset ---
pattern = "0123456789ABCDEF"      # synthetic, no copyright concerns
text = pattern * 200
chars = sorted(set(text))
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
data = torch.tensor([stoi[c] for c in text], dtype=torch.long)
vocab_size = len(chars)
max_seq_len = 32

model = TinyLM(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model built. Trainable parameters: {n_params:,}")

In [ ]:
# --- sanity check: one forward + backward pass before training ---
xb0 = data[:max_seq_len].unsqueeze(0).to(device)
yb0 = data[1:max_seq_len + 1].unsqueeze(0).to(device)
out0 = model(xb0)
logits0 = out0[0] if isinstance(out0, tuple) else out0
print(f"Sanity check -- logits shape: {tuple(logits0.shape)} (expect [1, {max_seq_len}, {vocab_size}])")
loss0 = F.cross_entropy(logits0.reshape(-1, vocab_size), yb0.reshape(-1))
if isinstance(out0, tuple):
    loss0 = loss0 + out0[1]
loss0.backward()
n_nan_grads = sum(torch.isnan(p.grad).any().item() for p in model.parameters() if p.grad is not None)
print(f"Sanity check -- initial loss: {loss0.item():.4f}, NaN grads: {n_nan_grads}")
model.zero_grad()

In [ ]:
# --- training loop ---
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

opt = torch.optim.AdamW(model.parameters(), lr=3e-3)
n_steps, batch_size = 300, 16
print("Training on synthetic periodic sequence (verifies grads flow end-to-end)...")
for step in range(n_steps):
    xb, yb = get_batch(data, max_seq_len, batch_size, device)
    out = model(xb)
    logits = out[0] if isinstance(out, tuple) else out
    loss = F.cross_entropy(logits.reshape(-1, vocab_size), yb.reshape(-1))
    if isinstance(out, tuple):
        loss = loss + out[1]
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()
    if step % 50 == 0 or step == n_steps - 1:
        print(f"  step {step:4d} | loss {loss.item():.4f}")

In [ ]:
# --- generation ---
@torch.no_grad()
def generate(model, start_idx, n_new):
    model.eval()
    idx = start_idx.clone()
    for _ in range(n_new):
        out = model(idx)
        logits = out[0] if isinstance(out, tuple) else out
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)
    model.train()
    return idx

start = data[:8].unsqueeze(0).to(device)
gen = generate(model, start, 48)[0].tolist()
print("Generated (should show visible periodicity if training worked):")
print(''.join(itos[i] for i in gen))

## Where to go from here

- **Visualize the gates.** Log the per-branch gate values over training and
  see which branch the model leans on for this toy periodic task — a purely
  local, highly repetitive pattern is a good test of whether the sliding
  window branch dominates, as you'd expect.
- **Try a real sparse dispatch** for the compressed and window branches too
  (not just the selected branch) — the current notebook computes those two
  branches densely for simplicity; skipping masked positions instead of
  computing and discarding them is where the paper's actual hardware
  speedup comes from.
- **Compare against MLA** (`../mla`) — a different DeepSeek idea for cheaper
  attention, this time by compressing what gets *cached* rather than
  reducing what gets *attended to*. NSA and MLA are complementary and are
  combined together in some DeepSeek models.

Reference: DeepSeek-AI, *"Native Sparse Attention: Hardware-Aligned and
Natively Trainable Sparse Attention,"* 2025.